# Intro to Model and Data Drift

Once a model is deployed, the data it sees can slowly change. Sometimes the feature distribution shifts, sometimes the target behavior changes, and sometimes both happen together. This notebook shows how to compare two taxi datasets and turn that intuition into measurable drift signals.


```mermaid
flowchart LR
    A["Training month"] --> C["Compare distributions"]
    B["New month"] --> C
    C --> D["Feature drift"]
    C --> E["Target drift"]
    C --> F["Prediction drift"]
    D --> G["Investigate or monitor"]
    E --> G
    F --> G
```


**Example:**

You've built a model that predicts how much a person will spend in an online webshop, using the age and the gender of a person. The training dataset mostly consists of women, with the average age of all customers being 35. The model learns that women and customers in their mid-thirties spend more. After monitoring the model's predictive performance, you notice that it's starting to decrease... But why? 

After doing some research you discover that the webshop now has younger customers: the average age has dropped from 35 years to 25 years. Furthermore, the live data shows that men, not women, are now the majority!

This change in the customer mix is data drift. It may help explain the observed performance drop, but a different feature distribution does not by itself prove that the relationship between the features and the target has changed.

**Data drift** means the input distribution changes. **Concept drift** means the relationship between inputs and the target changes. To establish performance degradation, check predictions against actual outcomes; feature or prediction drift alone is only a signal to investigate. See Evidently's [drift monitoring use cases](https://docs.evidentlyai.com/metrics/preset_data_drift#use-case).

Let's have a look at the NYC Green Taxi dataset.

In [ ]:
import os
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
from dotenv import load_dotenv

Let's start by loading the datasets. The first one is the dataset we train the model on:

In [ ]:
year = 2024
month = 1
color = "green"

df_2024_jan = pd.read_parquet(f"./data/{color}_tripdata_{year}-{month:02d}.parquet")

The second one is the dataset we compare the first one to:

In [ ]:
year = 2025
month = 8
color = "green"

data_path = Path(f"./data/{color}_tripdata_{year}-{month:02d}.parquet")
data_path.parent.mkdir(exist_ok=True)

if not data_path.exists():
    urlretrieve(
        f"https://d37ci6vzurychx.cloudfront.net/trip-data/{color}_tripdata_{year}-{month:02d}.parquet",
        data_path,
    )

df_2025_aug = pd.read_parquet(data_path)

In [ ]:
features = ["PULocationID", "DOLocationID", "trip_distance"]
target = "duration"

In [ ]:
def preprocess(df):
    df["duration"] = (
        df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]
    ).dt.total_seconds() / 60
    df = df[(df["duration"] >= 1) & (df["duration"] <= 60)]
    df = df[df["trip_distance"].between(0.1, 100)]
    df[["PULocationID", "DOLocationID"]] = df[["PULocationID", "DOLocationID"]].astype(
        str
    )
    df = df[features + [target]]
    return df

The two months have different numbers of trips, so we randomly sample the larger one down to the size of the smaller one. This keeps the histograms below on the same scale. A fixed random seed makes the sample the same every time you run the notebook.

In [ ]:
df_2024_jan = preprocess(df_2024_jan)
df_2025_aug = preprocess(df_2025_aug)

if len(df_2024_jan) < len(df_2025_aug):
    print("Trimming df_2025_aug to match length of df_2024_jan.")
    df_2025_aug = df_2025_aug.sample(frac=1, random_state=42).reset_index(drop=True)
    df_2025_aug = df_2025_aug.iloc[: len(df_2024_jan)]
elif len(df_2024_jan) > len(df_2025_aug):
    print("Trimming df_2024_jan to match length of df_2025_aug.")
    df_2024_jan = df_2024_jan.sample(frac=1, random_state=42).reset_index(drop=True)
    df_2024_jan = df_2024_jan.iloc[: len(df_2025_aug)]
else:
    print("Both DataFrames have the same length")

### Compare the two months

In [ ]:
df_2024_jan.describe()

In [ ]:
df_2025_aug.describe()

In [ ]:
columns = df_2024_jan.columns
fig, axes = plt.subplots(1, len(columns), figsize=(16, 3.5))

for ax, var in zip(axes, columns):
    # The location IDs are stored as strings for the model; plot them as numbers.
    jan_values = pd.to_numeric(df_2024_jan[var])
    aug_values = pd.to_numeric(df_2025_aug[var])
    combined = pd.concat([jan_values, aug_values])
    # Use the same bins for both months and skip the most extreme 1% of values,
    # so a few outliers don't squeeze the rest of the histogram.
    value_range = (combined.quantile(0.005), combined.quantile(0.995))
    ax.hist(
        jan_values, bins=40, range=value_range, alpha=0.4, label="Jan 2024 (training)"
    )
    ax.hist(
        aug_values,
        bins=40,
        range=value_range,
        histtype="step",
        linewidth=1.5,
        label="Aug 2025 (current)",
    )
    ax.set_title(var)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2)
fig.tight_layout(rect=(0, 0, 1, 0.9))
plt.show()

The August trips are longer: `trip_distance` and `duration` both shift to the right, and some pickup and dropoff zones are busier or quieter than in January.

We can also measure the difference between the two datasets with:

- **Jensen-Shannon distance** for categorical features,
- **Wasserstein distance** for numerical features.

For categorical features, we first convert location IDs into normalized frequency distributions, then compare those distributions. The lower the value, the more similar the two datasets are.


Let's calculate the distance between the features of the two datasets:


In [ ]:
from scipy.spatial.distance import jensenshannon
from scipy.stats import wasserstein_distance


def drift_jensenshannon_test(var_train: pd.Series, var_test: pd.Series) -> float:
    """Measure distance between categorical probability distributions."""
    categories = sorted(set(var_train.dropna()) | set(var_test.dropna()))
    train_distribution = var_train.value_counts(normalize=True).reindex(
        categories, fill_value=0
    )
    test_distribution = var_test.value_counts(normalize=True).reindex(
        categories, fill_value=0
    )
    return float(jensenshannon(train_distribution, test_distribution))


def drift_wasserstein_test(var_train: pd.Series, var_test: pd.Series) -> float:
    """Measure distance between two numerical distributions."""
    return float(wasserstein_distance(var_train, var_test))

In [ ]:
df_drift = pd.DataFrame(
    {
        "feature": pd.Series(dtype="object"),
        "distance": pd.Series(dtype="float64"),
        "type": pd.Series(dtype="object"),
    }
)

cat_features = ["PULocationID", "DOLocationID"]
num_features = ["trip_distance"]

for feature in num_features:
    df_drift = pd.concat(
        [
            df_drift,
            pd.DataFrame(
                [
                    {
                        "feature": feature,
                        "distance": drift_wasserstein_test(
                            df_2024_jan[feature], df_2025_aug[feature]
                        ),
                        "type": "Wasserstein",
                    }
                ]
            ),
        ],
        ignore_index=True,
    )

for feature in cat_features:
    df_drift = pd.concat(
        [
            df_drift,
            pd.DataFrame(
                [
                    {
                        "feature": feature,
                        "distance": drift_jensenshannon_test(
                            df_2024_jan[feature], df_2025_aug[feature]
                        ),
                        "type": "Jensen-Shannon",
                    }
                ]
            ),
        ],
        ignore_index=True,
    )

There are more **tests to detect drift**, but this is a good start. If you want to learn more about **drift detection**, check out Evidently's [guide to drift detection on large datasets](https://www.evidentlyai.com/blog/data-drift-detection-large-datasets).

Keep in mind that the two distances use different scales: the Jensen-Shannon distance is always between 0 and 1, while the Wasserstein distance is measured in the feature's own units, miles for `trip_distance` and minutes for `duration`. A single threshold for both is a simplification. In practice, thresholds should be calibrated for the data and business context. For this lesson, we use `0.1` as a simple rule of thumb so we can practice turning distances into a drift flag:

In [ ]:
df_drift["True/False Drift"] = df_drift["distance"] > 0.1
df_drift

This confirms what we saw in the histograms: all three features have drifted.

Using this method **we can also detect drift in the target variable**. Let's have a look at the target variable:

In [ ]:
drift_wasserstein_test(df_2024_jan[target], df_2025_aug[target])

The target has drifted too: its distance is well above 0.1.

We can also check whether the model predictions themselves are drifting. For that step we load the registered model created in notebook 01 and score both datasets with the same feature set.


In [ ]:
# Load environment variables from .env file
load_dotenv()

# Get the MLflow tracking URI from environment variables
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "sqlite:///mlflow.db")

# Set up the connection to MLflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("green-taxi-monitoring")

In [ ]:
def load_model(model_name):
    alias = "production"
    model_uri = f"models:/{model_name}@{alias}"
    model = mlflow.pyfunc.load_model(model_uri)
    return model

In [ ]:
model_name = "green-taxi-ride-duration"

loaded_model = load_model(model_name)

In [ ]:
df_2024_jan["prediction"] = loaded_model.predict(
    df_2024_jan[features].to_dict(orient="records")
)

In [ ]:
df_2025_aug["prediction"] = loaded_model.predict(
    df_2025_aug[features].to_dict(orient="records")
)

Now we can calculate the drift in the predictions:

In [ ]:
drift_wasserstein_test(df_2024_jan["prediction"], df_2025_aug["prediction"])

And also here we can see drift. That does not automatically mean the model is unusable, but it is a strong signal that the training distribution and the current production-like batch are no longer aligned. In practice, this is the kind of evidence that can trigger retraining or a closer investigation.


In the next notebook we will use Evidently to automate these checks and package them into a reusable report.
